# SemEval 2026 Task 13 — End-to-End LightGBM Submission

Students:
1. Hoang Trong Vu - 25C15028
2. To Tan Hiep - 25C11039
3. Nguyen Minh Quang - 25C15057

This notebook reproduces the LB-champion submission:

- **Feature set**:
  - Text surface features
  - NPR score using Qwen-2.5-Coder-1.5B
  - AST-reated feature
- **Normalization**: per-language for train and validation set, global normalization for test sample and test sets.
- **Model**: LightGBM (deterministic, `random_state=42`).

Due to the environment mismatch between Kaggle and our setting, the precomputed features will be download. In this notebook, we use the data to train models, make predictions.

The pipeline is fully deterministic — running this notebook end-to-end
yields the exact same submission CSV every time.


In [1]:
!pip install -q lightgbm==4.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.6 MB/s eta 0:00:00


## Configuration


In [2]:
import os

# ── Text and AST features precomputed parquet URLs
DATA_URLS = {
    "train":       "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/train_ood_features.parquet",
    "val":         "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/val_ood_features.parquet",
    "test_sample": "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/test_sample_ood_features.parquet",
    "test":        "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/test_ood_features.parquet",
}

# ── NPR precomputed parquet URLs
NPR_URLS = {
    "train":       "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/train_npr_qwen.parquet",
    "val":         "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/val_npr_qwen.parquet",
    "test_sample": "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/test_sample_npr_qwen.parquet",
    "test":        "https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/test_npr_qwen.parquet",
}

# ── Working directories on Kaggle ────────────────────────────────────────────
WORK_DIR    = "/kaggle/working"
DATA_DIR    = os.path.join(WORK_DIR, "features")
NPR_DIR     = os.path.join(WORK_DIR, "npr")
SUB_PATH    = os.path.join(WORK_DIR, "submission.csv")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(NPR_DIR, exist_ok=True)

# ── Determinism ──────────────────────────────────────────────────────────────
import numpy as np
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)


## Download precomputed features

### Text and AST Features

In [3]:
for split, url in DATA_URLS.items():
    out = os.path.join(DATA_DIR, f"{split}_ood_features.parquet")
    if os.path.exists(out):
        print(f"  [skip] {out} already present")
        continue
    print(f"  [wget] {url} -> {out}")
    !wget -q -O "$out" "$url"
    assert os.path.exists(out) and os.path.getsize(out) > 0, f"download failed: {out}"

print("\nFeatures parquets ready:", os.listdir(DATA_DIR))


  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/train_ood_features.parquet -> /kaggle/working/features/train_ood_features.parquet
  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/val_ood_features.parquet -> /kaggle/working/features/val_ood_features.parquet
  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/test_sample_ood_features.parquet -> /kaggle/working/features/test_sample_ood_features.parquet
  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/proccesed_data/test_ood_features.parquet -> /kaggle/working/features/test_ood_features.parquet

Features parquets ready: ['train_ood_features.parquet', 'val_ood_features.parquet', 'test_sample_ood_features.parquet', 'test_ood_features.parquet']


### NPR scores:
NPR scoring is GPU-bound (~1-2 s per sample on CUDA, days on CPU). To make this
notebook runnable end-to-end on a Kaggle CPU instance we download precomputed
scores instead of recomputing them.

In [4]:
for split, url in NPR_URLS.items():
    out = os.path.join(NPR_DIR, f"{split}_npr_qwen.parquet")
    if os.path.exists(out):
        print(f"  [skip] {out} already present")
        continue
    print(f"  [wget] {url} -> {out}")
    !wget -q -O "$out" "$url"
    assert os.path.exists(out) and os.path.getsize(out) > 0, f"download failed: {out}"

print("\nNPR parquets ready:", os.listdir(NPR_DIR))


  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/train_npr_qwen.parquet -> /kaggle/working/npr/train_npr_qwen.parquet
  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/val_npr_qwen.parquet -> /kaggle/working/npr/val_npr_qwen.parquet
  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/test_sample_npr_qwen.parquet -> /kaggle/working/npr/test_sample_npr_qwen.parquet
  [wget] https://storage.googleapis.com/snapedit-dev-visualization/vuhoang/tmp/semeval_data_precomputed/test_npr_qwen.parquet -> /kaggle/working/npr/test_npr_qwen.parquet

NPR parquets ready: ['train_npr_qwen.parquet', 'test_sample_npr_qwen.parquet', 'val_npr_qwen.parquet', 'test_npr_qwen.parquet']


## Load features data


In [5]:
TRAIN_PARQUET      = os.path.join(DATA_DIR, "train_ood_features.parquet")
VAL_PARQUET        = os.path.join(DATA_DIR, "val_ood_features.parquet")
TEST_SAMPLE_PARQUET = os.path.join(DATA_DIR, "test_sample_ood_features.parquet")
TEST_PARQUET       = os.path.join(DATA_DIR, "test_ood_features.parquet")


In [6]:
import pandas as pd
import time

t0 = time.time()
df_train = pd.read_parquet(TRAIN_PARQUET)
df_val   = pd.read_parquet(VAL_PARQUET)
df_ts    = pd.read_parquet(TEST_SAMPLE_PARQUET)
df_test  = pd.read_parquet(TEST_PARQUET)
print(f"loaded all parquets in {time.time()-t0:.1f}s")

print(f"train       : {len(df_train):>8,} rows | columns: {list(df_train.columns)[:6]}…")
print(f"val         : {len(df_val):>8,} rows")
print(f"test_sample : {len(df_ts):>8,} rows  (labelled OOD proxy)")
print(f"test        : {len(df_test):>8,} rows  (unlabelled — submission target)")

print("\ntrain language dist       :", dict(df_train['language'].value_counts()))
print("val language dist         :", dict(df_val['language'].value_counts()))
print("test_sample language dist :", dict(df_ts['language'].value_counts()))
print("\ntrain label dist          :", dict(df_train['label'].value_counts()))
print("val label dist            :", dict(df_val['label'].value_counts()))
print("test_sample label dist    :", dict(df_ts['label'].value_counts()))


loaded all parquets in 7.5s
train       :  500,000 rows | columns: ['code', 'generator', 'label', 'language', 'char_entropy', 'bigram_char_entropy']…
val         :  100,000 rows
test_sample :    1,000 rows  (labelled OOD proxy)
test        :  500,000 rows  (unlabelled — submission target)

train language dist       : {'Python': np.int64(457306), 'C++': np.int64(23392), 'Java': np.int64(19302)}
val language dist         : {'Python': np.int64(91461), 'C++': np.int64(4679), 'Java': np.int64(3860)}
test_sample language dist : {'Python': np.int64(303), 'Java': np.int64(256), 'C#': np.int64(122), 'JavaScript': np.int64(85), 'C++': np.int64(75), 'Go': np.int64(60), 'C': np.int64(51), 'PHP': np.int64(48)}

train label dist          : {1: np.int64(261525), 0: np.int64(238475)}
val label dist            : {1: np.int64(52305), 0: np.int64(47695)}
test_sample label dist    : {0: np.int64(777), 1: np.int64(223)}


In [7]:
# Only the columns we actually need for `minimal_shift_npr_ast4`.
TEXT_COLS = ["comment_ratio", "trailing_whitespace_ratio",
             "blank_line_gap_std", "comment_word_avg", "burstiness"]
TREE_COLS = ["maintainability_index"]

LANGUAGE = ["language"]
LABEL = ["label"]
ID = ["ID"]

df_train = df_train[TEXT_COLS + TREE_COLS + LABEL + LANGUAGE]
df_val = df_val[TEXT_COLS + TREE_COLS + LABEL + LANGUAGE]
df_ts = df_ts[TEXT_COLS + TREE_COLS + LABEL + LANGUAGE]
df_test = df_test[TEXT_COLS + TREE_COLS + ID]

In [8]:
df_train.head()

,comment_ratio,trailing_whitespace_ratio,blank_line_gap_std,comment_word_avg,burstiness,maintainability_index,label,language
0,0.000000,0.0,0.000000,0.0,4.624224,57.522308,0,Python
1,0.000000,0.0,1.316957,0.0,25.367464,39.256268,1,Python
2,0.285714,0.0,1.224745,5.5,0.207959,57.614742,1,Python
3,0.000000,0.0,0.000000,0.0,7.377419,55.610817,0,Python
4,0.000000,0.0,0.000000,0.0,6.575862,50.495399,0,Python


In [9]:
df_test.head()

,comment_ratio,trailing_whitespace_ratio,blank_line_gap_std,comment_word_avg,burstiness,maintainability_index,ID
0,0.0,0.000000,0.000000,0.0,9.720657,52.036362,0
2,0.0,0.000000,0.000000,0.0,6.299528,64.499115,2
5,0.0,0.017857,3.685557,0.0,30.894346,38.269989,5
6,0.0,0.000000,0.000000,0.0,4.041667,56.266087,6
7,0.0,0.000000,3.141627,0.0,7.264802,28.062696,7


## Merge NPR scores

The NPR parquets must have the same row order as the corresponding
handcraft parquets — we merge by row index.


In [10]:
def merge_npr(df, split_name):
    npr_path = os.path.join(NPR_DIR, f"{split_name}_npr_qwen.parquet")
    df_npr = pd.read_parquet(npr_path, columns=["npr_rank_qwen"])
    assert len(df_npr) == len(df), (
        f"NPR parquet has {len(df_npr)} rows but {split_name} has {len(df)}"
    )
    df = df.copy()
    df["npr_rank_qwen"] = df_npr["npr_rank_qwen"].values.astype("float32")
    return df

df_train = merge_npr(df_train, "train")
df_val   = merge_npr(df_val,   "val")
df_ts    = merge_npr(df_ts,    "test_sample")
df_test  = merge_npr(df_test,  "test")
print("NPR merged on all 4 splits.")
print(f"  train       npr_rank_qwen  mean={df_train['npr_rank_qwen'].mean():.4f}  std={df_train['npr_rank_qwen'].std():.4f}")
print(f"  val         npr_rank_qwen  mean={df_val['npr_rank_qwen'].mean():.4f}  std={df_val['npr_rank_qwen'].std():.4f}")
print(f"  test_sample npr_rank_qwen  mean={df_ts['npr_rank_qwen'].mean():.4f}  std={df_ts['npr_rank_qwen'].std():.4f}")
print(f"  test        npr_rank_qwen  mean={df_test['npr_rank_qwen'].mean():.4f}  std={df_test['npr_rank_qwen'].std():.4f}")


NPR merged on all 4 splits.
  train       npr_rank_qwen  mean=1.4014  std=0.3218
  val         npr_rank_qwen  mean=1.4004  std=0.3185
  test_sample npr_rank_qwen  mean=1.3260  std=0.2702
  test        npr_rank_qwen  mean=1.3293  std=0.2679


In [11]:
df_train.head()

,comment_ratio,trailing_whitespace_ratio,blank_line_gap_std,comment_word_avg,burstiness,maintainability_index,label,language,npr_rank_qwen
0,0.000000,0.0,0.000000,0.0,4.624224,57.522308,0,Python,1.490291
1,0.000000,0.0,1.316957,0.0,25.367464,39.256268,1,Python,0.991226
2,0.285714,0.0,1.224745,5.5,0.207959,57.614742,1,Python,1.252980
3,0.000000,0.0,0.000000,0.0,7.377419,55.610817,0,Python,1.296901
4,0.000000,0.0,0.000000,0.0,6.575862,50.495399,0,Python,1.415567


## Per-language normalization + LightGBM training

1. Fit one `StandardScaler` per language on the **train** handcrafted features.
2. Apply per-language scalers to **train** + **val**; rows whose language is
   not in train fall back to a global scaler.
3. Apply the **global** scaler to **test_sample** + **test** (test has no
   `language` column at all).
4. Train LightGBM with `random_state=42`, deterministic settings, and
   `early_stopping(100)` on val AUC.


In [12]:
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

FEATURE_COLUMNS = [
    "comment_ratio",
    "trailing_whitespace_ratio",
    "blank_line_gap_std",
    "comment_word_avg",
    "burstiness",
    "npr_rank_qwen",
    "maintainability_index",
]
LABEL_COL = "label"
ID_COL = "ID"

assert all(c in df_train.columns for c in FEATURE_COLUMNS + [LABEL_COL]), \
    f"missing columns in train: {set(FEATURE_COLUMNS + [LABEL_COL]) - set(df_train.columns)}"
assert ID_COL in df_test.columns, f"test parquet missing `{ID_COL}` column"


# ── per-language scaler fitting ──────────────────────────────────────────────

def fit_per_language_scalers(X_train, languages_train):
    per_lang = {}
    langs = pd.Series(languages_train).fillna("__NA__").values
    for lang in np.unique(langs):
        mask = (langs == lang)
        if mask.sum() < 2:
            continue
        sc = StandardScaler()
        sc.fit(X_train[mask])
        per_lang[lang] = sc
    global_scaler = StandardScaler().fit(X_train)
    return per_lang, global_scaler


def transform_per_language(X, languages, per_lang, global_scaler):
    X = np.asarray(X, dtype=np.float32)
    out = np.empty_like(X)
    langs = pd.Series(languages).fillna("__NA__").values
    fb = 0
    for lang in np.unique(langs):
        mask = (langs == lang)
        sc = per_lang.get(lang, global_scaler)
        if lang not in per_lang:
            fb += int(mask.sum())
        out[mask] = sc.transform(X[mask])
    if fb:
        print(f"    [norm] {fb} rows used global fallback (unseen language)")
    return out


# ── assemble matrices ────────────────────────────────────────────────────────

def fillna_to_float32(df):
    return df[FEATURE_COLUMNS].fillna(0.0).astype(np.float32).values

X_tr_raw = fillna_to_float32(df_train)
X_va_raw = fillna_to_float32(df_val)
X_ts_raw = fillna_to_float32(df_ts)
X_te_raw = fillna_to_float32(df_test)

y_tr = df_train[LABEL_COL].values.astype(int)
y_va = df_val[LABEL_COL].values.astype(int)
y_ts = df_ts[LABEL_COL].values.astype(int)

per_lang, gscaler = fit_per_language_scalers(X_tr_raw, df_train["language"].values)
print("Per-language scalers fitted for:", sorted(per_lang.keys()))
for lang in sorted(per_lang.keys()):
    n = (df_train["language"].values == lang).sum()
    print(f"  {lang:<10}  fit on {n:>8,} train rows")

X_tr = transform_per_language(X_tr_raw, df_train["language"].values, per_lang, gscaler)
X_va = transform_per_language(X_va_raw, df_val["language"].values,   per_lang, gscaler)
# test_sample + test use the GLOBAL scaler (test has no `language` column,
# test_sample has many languages unseen in train).
X_ts = gscaler.transform(X_ts_raw)
X_te = gscaler.transform(X_te_raw)

print(f"\n  train       {X_tr.shape}")
print(f"  val         {X_va.shape}")
print(f"  test_sample {X_ts.shape}  (per-lang scaler reaches  {(pd.Series(df_ts['language']).isin(per_lang)).sum()} / {len(df_ts)} rows; remainder forced to global)")
print(f"  test        {X_te.shape}  (global scaler — no language column)")


# ── train LightGBM (LB-champion config) ──────────────────────────────────────

print("\n" + "=" * 60)
print("Training LightGBM (LB-champion config, deterministic, seed=42)")
print("=" * 60)
t0 = time.time()
model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    metric="auc",
    random_state=SEED,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[log_evaluation(200), early_stopping(100)],
)
print(f"\n  best_iteration: {model.best_iteration_}")
print(f"  train_time    : {time.time()-t0:.1f}s")


Per-language scalers fitted for: ['C++', 'Java', 'Python']
  C++         fit on   23,392 train rows
  Java        fit on   19,302 train rows
  Python      fit on  457,306 train rows

  train       (500000, 7)
  val         (100000, 7)
  test_sample (1000, 7)  (per-lang scaler reaches  634 / 1000 rows; remainder forced to global)
  test        (500000, 7)  (global scaler — no language column)

Training LightGBM (LB-champion config, deterministic, seed=42)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.920264
[400]	valid_0's auc: 0.921806
[600]	valid_0's auc: 0.922403
[800]	valid_0's auc: 0.922606
[1000]	valid_0's auc: 0.92272
Early stopping, best iteration is:
[955]	valid_0's auc: 0.922743

  best_iteration: 955
  train_time    : 30.0s


## Predict and write submission CSV


In [13]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
)

def report(name, y_true, prob):
    pred = (prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, pred)
    f1m = f1_score(y_true, pred, average="macro")
    auc = roc_auc_score(y_true, prob)
    print(f"\n[{name}]  accuracy={acc:.4f}  macro_f1={f1m:.4f}  roc_auc={auc:.4f}")
    print(f"  predicted AI rate: {pred.mean():.4f}   true AI rate: {(y_true==1).mean():.4f}")
    print(f"  confusion matrix (rows=true, cols=pred):")
    cm = confusion_matrix(y_true, pred)
    print(f"            pred=Human  pred=AI")
    print(f"  Human  : {cm[0,0]:>10,d}  {cm[0,1]:>7,d}")
    print(f"  AI     : {cm[1,0]:>10,d}  {cm[1,1]:>7,d}")
    print(f"  classification_report:")
    print(classification_report(y_true, pred, target_names=["Human", "AI"], digits=4))

# ── eval on val ──
print("=" * 60)
print("Evaluation")
print("=" * 60)
prob_va = model.predict_proba(X_va)[:, 1]
report("VAL (in-distribution)", y_va, prob_va)

# ── eval on test_sample (the labelled OOD proxy) ──
prob_ts = model.predict_proba(X_ts)[:, 1]
report("TEST_SAMPLE (OOD proxy, multilingual)", y_ts, prob_ts)

# ── final test prediction → submission ──
print("\n" + "=" * 60)
print("Generating final submission")
print("=" * 60)
prob_te = model.predict_proba(X_te)[:, 1]
pred_te = (prob_te >= 0.5).astype(int)

submission = pd.DataFrame({
    ID_COL: df_test[ID_COL].values,
    LABEL_COL: pred_te.astype(int),
})
submission.to_csv(SUB_PATH, index=False)
print(f"Submission written -> {SUB_PATH}")
print(f"  rows         : {len(submission):,}")
print(f"  predicted AI : {pred_te.sum():,}  ({pred_te.mean():.4%})")
print(f"  predicted H  : {(pred_te==0).sum():,}  ({(pred_te==0).mean():.4%})")
print()
print("first 5 rows of submission:")
print(submission.head())


Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



[VAL (in-distribution)]  accuracy=0.8575  macro_f1=0.8574  roc_auc=0.9227
  predicted AI rate: 0.4501   true AI rate: 0.5231
  confusion matrix (rows=true, cols=pred):
            pred=Human  pred=AI
  Human  :     44,220    3,475
  AI     :     10,773   41,532
  classification_report:
              precision    recall  f1-score   support

       Human     0.8041    0.9271    0.8612     47695
          AI     0.9228    0.7940    0.8536     52305

    accuracy                         0.8575    100000
   macro avg     0.8634    0.8606    0.8574    100000
weighted avg     0.8662    0.8575    0.8572    100000


[TEST_SAMPLE (OOD proxy, multilingual)]  accuracy=0.7310  macro_f1=0.6710  roc_auc=0.7702
  predicted AI rate: 0.3500   true AI rate: 0.2230
  confusion matrix (rows=true, cols=pred):
            pred=Human  pred=AI
  Human  :        579      198
  AI     :         71      152
  classification_report:
              precision    recall  f1-score   support

       Human     0.8908   

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Submission written -> /kaggle/working/submission.csv
  rows         : 500,000
  predicted AI : 182,359  (36.4718%)
  predicted H  : 317,641  (63.5282%)

first 5 rows of submission:
   ID  label
0   0      0
1   2      0
2   5      1
3   6      0
4   7      0
